In [1]:
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

C:\Users\pramo\AppData\Local\Temp\ipykernel_3604\2961514931.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import WikipediaQueryRun


In [11]:
api_wrapper = WikipediaAPIWrapper(
    top_k_results=1,doc_content_chars_max=200
)

wiki_tool = WikipediaQueryRun(api_wrapper=api_wrapper)


In [12]:
from langchain_community.document_loaders import WebBaseLoader
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = WebBaseLoader("https://docs.smith.langchain.com/")
web_doc = loader.load()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 250
)
chunks = text_splitter.split_documents(web_doc)
# embeddings = OllamaEmbeddings(model="nano")
vector_store = FAISS.from_documents(chunks, OllamaEmbeddings(model="nomic-embed-text"))
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000023F739BEAD0>, search_kwargs={'k': 3})

In [13]:
from langchain_core.tools import create_retriever_tool

langsmith_tool = create_retriever_tool(
    retriever,
    "langsmith_search",
    "search for information about langsmith. For any question about langsmith, you must use this tool"
)

In [14]:
from langchain_community.utilities import ArxivAPIWrapper
from langchain_community.tools import ArxivQueryRun

arxiv_wrapper = ArxivAPIWrapper(top_k_results=1,doc_content_search_max = 200)
arxiv_tool = ArxivQueryRun(api_wrapper=arxiv_wrapper)


In [15]:
tools=[
    wiki_tool,
    langsmith_tool,
    arxiv_tool
]

In [46]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0)
print(type(llm))

<class 'langchain_groq.chat_models.ChatGroq'>


In [28]:
##pulling promts from langchain hub
from langsmith import Client

client = Client()

prompt = client.pull_prompt(
    "hwchase17/openai-functions-agent",
    dangerously_pull_public_prompt=True
)
prompt.messages

[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a helpful assistant'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='chat_history', optional=True),
 HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={}),
 MessagesPlaceholder(variable_name='agent_scratchpad')]

In [49]:
##creating agents 
from langchain.agents import create_agent
system_prompt = prompt.messages[0].prompt.template
agent = create_agent(
    model = llm,
    tools = [],
    system_prompt = system_prompt
)
for tool in tools:
    print(tool)

api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\pramo\\OneDrive\\Desktop\\ChatBot\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)
name='langsmith_search' description='search for information about langsmith. For any question about langsmith, you must use this tool' args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'> func=<function create_retriever_tool.<locals>.func at 0x0000023F73A079C0> coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000023F73A04360>
api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.UnexpectedEmptyPageError'>, <class 'arxiv.HTTPError'>), top_k_results=1, ARXIV_MAX_QUERY_LENGTH=300, continue_on_failure=False, load_max_docs=100, load_all_available_meta=False, doc_content_chars_max=4000)


In [50]:
res = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me about LangSmith"
            }
        ]
    }
)
print(res)

{'messages': [HumanMessage(content='Tell me about LangSmith', additional_kwargs={}, response_metadata={}, id='52a58829-8541-4bed-8277-46a29e593c3a'), AIMessage(content='I couldn\'t find any information on "LangSmith." It\'s possible that it\'s a lesser-known or fictional entity, or it could be a misspelling or variation of a different name.\n\nHowever, I did find information on "LangSmith" being a surname. If you\'re looking for information on someone with this surname, could you please provide more context or details about who LangSmith is or what they are known for?\n\nAlternatively, I can also suggest some possible alternatives or related information:\n\n- Langston Hughes: An American poet, novelist, and playwright who was a central figure in the Harlem Renaissance movement.\n- Langley Smith: A fictional character from the TV show "The Vampire Diaries."\n- Smith Lang: A fictional character from the TV show "The Walking Dead."\n\nIf none of these options match what you\'re looking fo

In [39]:
##agent executor 
from langchain_classic.agents import AgentExecutor
agent_exe = AgentExecutor(agent = agent,tools=tools,verbose=True)
agent_exe

AgentExecutor(verbose=True, agent=RunnableAgent(runnable=<langgraph.graph.state.CompiledStateGraph object at 0x0000023F7665C410>, input_keys_arg=[], return_keys_arg=[], stream_runnable=True), tools=[WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Users\\pramo\\OneDrive\\Desktop\\ChatBot\\.venv\\Lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=1, lang='en', load_all_available_meta=False, doc_content_chars_max=200)), StructuredTool(name='langsmith_search', description='search for information about langsmith. For any question about langsmith, you must use this tool', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x0000023F73A079C0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x0000023F73A04360>), ArxivQueryRun(api_wrapper=ArxivAPIWrapper(arxiv_search=<class 'arxiv.Search'>, arxiv_exceptions=(<class 'arxiv.ArxivError'>, <class 'arxiv.Unexpecte